# 05 Cross Validation and Hyperparameter Optimisation

This notebook evaluates model reliability through cross-validation and improves selected algorithms through hyperparameter optimisation. The objective is to move beyond a single train-test split and estimate how consistently each model performs across different partitions of the data.

Cross-validation, GridSearchCV, greedy search, and Optuna address different parts of the modelling process. Cross-validation measures stability, GridSearchCV exhaustively evaluates a defined parameter grid, greedy search tunes one parameter at a time, and Optuna performs guided probabilistic search over larger or more flexible spaces. Together, these methods reduce the risk of selecting a model that performs well only by chance.


In [1]:
import os
import sys
import time

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(os.path.abspath('..'))

RANDOM_STATE = 42
SAMPLE_SIZE = 50000
RATING_DTYPES = {
    'username': 'category',
    'anime_id': 'int32',
    'status': 'category',
    'score': 'float32',
    'is_rewatching': 'float32',
    'num_watched_episodes': 'float32',
}

%matplotlib inline
sns.set_theme(style='whitegrid')


In [2]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

from src.hyperopt import greedy_search, grid_search, optuna_study
from src.models import get_models
from src.preprocessing import build_preprocess
from src.evaluate import compute_classification_metrics
from src.train import evaluate_model

SKIP_OPTIMIZATION = ['SVC_linear', 'SVC_rbf', 'SVC_poly', 'MLPClassifier_single', 'MLPClassifier_multi']
OPTIMIZATION_REGRESSION_MODELS = [
    'Linear', 'Ridge', 'Lasso', 'KNN', 'DecisionTree'
]
OPTIMIZATION_CLASSIFICATION_MODELS = [
    'NaiveBayes', 'KNNClassifier', 'DecisionTreeClassifier'
]


In [3]:
def make_pipeline(df, target, model):
    return Pipeline([
        ('preprocess', build_preprocess(df, target)),
        ('model', model),
    ])


In [4]:
df = pd.read_csv('../datasets/ratings.csv', dtype=RATING_DTYPES)
df = df[df['score'] > 0]
df = df[df['num_watched_episodes'] <= 10000]
df = df[df['num_watched_episodes'] >= 0]
df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

TARGET = 'score'
X_df = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=RANDOM_STATE
)
models = get_models('regression')


## Cross Validation

Cross-validation is necessary because a single train-test split may produce optimistic or pessimistic results depending on which observations happen to fall into each subset. In 5-fold cross-validation, the dataset is divided into five parts; each part is used once as validation while the remaining four parts are used for training. The final mean RMSE estimates average generalisation performance across folds.

This procedure helps protect against overfitting because models must perform well across multiple validation sets rather than only one. It also reduces the risk of accidental data leakage when preprocessing is included inside the pipeline, because each fold fits preprocessing steps only on its training portion. This is especially important for imputation, scaling, and one-hot encoding.

The CV RMSE mean indicates expected prediction error on unseen data, while the CV RMSE standard deviation indicates stability. A low standard deviation means performance is consistent across folds. A high standard deviation suggests that the model is sensitive to the particular training sample, possibly because of class/user heterogeneity, sparse categories, or overfitting. Models with similar mean RMSE should therefore be compared using their standard deviation and training cost as well.


In [5]:
cv_results = []
while mlflow.active_run():
    mlflow.end_run()
with mlflow.start_run(nested=True, run_name='05_cross_validation'):
    for name, model in models.items():
        pipeline_model = make_pipeline(df, TARGET, model)
        scores = cross_val_score(
            pipeline_model,
            X_df,
            y,
            cv=5,
            scoring='neg_mean_squared_error',
            n_jobs=1,
        )
        rmse_scores = np.sqrt(-scores)
        cv_results.append({
            'model': name,
            'cv_rmse_mean': rmse_scores.mean(),
            'cv_rmse_std': rmse_scores.std(),
        })

df_cv = pd.DataFrame(cv_results).sort_values('cv_rmse_mean')
display(df_cv)


c:\Users\joaop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\joaop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\joaop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\joaop\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=500).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users

,model,cv_rmse_mean,cv_rmse_std
9,GradientBoosting,1.716795,0.009377
8,RandomForest,1.719488,0.009296
7,DecisionTree,1.722567,0.009020
10,MLP_single,1.723333,0.009639
11,MLP_multi,1.725978,0.006898
1,Ridge,1.731979,0.011050
0,Linear,1.731980,0.011052
2,Lasso,1.732127,0.011106
6,KNN,1.834152,0.046657
4,SVR_rbf,2.577730,0.006562


## GridSearch: Ridge, Lasso, K-NN Regressor, and K-NN Classifier

GridSearchCV performs exhaustive search over a predefined set of hyperparameter values. It is appropriate when the number of candidate combinations is small and when each value has a clear interpretation. In this notebook, Ridge and Lasso are tuned through `alpha`, while K-NN is tuned through `n_neighbors` and weighting strategy.

For Ridge and Lasso, the best alpha should be compared with the manual alpha analysis from Notebook 02. If both analyses identify similar alpha ranges, the regularisation behaviour is stable. A low alpha indicates that little shrinkage is needed, while a higher alpha indicates that stronger regularisation improves generalisation. The interpretation follows the bias-variance tradeoff: increasing alpha reduces variance but can increase bias if the model becomes too constrained.

For K-NN regression, the selected K indicates the neighbourhood size that minimises validation RMSE. The K-NN classifier uses the same principle but optimises F1 for the binary high-score target. The best classifier K does not have to match the best regression K because regression averages numeric scores, while classification predicts class membership and depends on the class boundary around `score >= 8`. Differences between the two optimal K values reveal that the continuous and binary tasks have different local structures.


In [6]:
ridge_grid = {'model__alpha': [0.01, 0.1, 1, 10, 100]}
lasso_grid = {'model__alpha': [0.001, 0.01, 0.1, 1, 10]}
knn_grid = {'model__n_neighbors': list(range(3, 32, 2)), 'model__weights': ['uniform', 'distance']}

while mlflow.active_run():
    mlflow.end_run()
with mlflow.start_run(nested=True, run_name='05_grid_search_regression'):
    best_ridge = grid_search(make_pipeline(df, TARGET, models['Ridge']), ridge_grid, X_train_df, y_train)
    best_lasso = grid_search(make_pipeline(df, TARGET, models['Lasso']), lasso_grid, X_train_df, y_train)
    best_knn = grid_search(make_pipeline(df, TARGET, models['KNN']), knn_grid, X_train_df, y_train)

grid_summary = pd.DataFrame([
    {'model': 'Ridge', **evaluate_model('Ridge_grid', best_ridge, X_test_df, y_test)},
    {'model': 'Lasso', **evaluate_model('Lasso_grid', best_lasso, X_test_df, y_test)},
    {'model': 'KNN', **evaluate_model('KNN_grid', best_knn, X_test_df, y_test), 'best_k': best_knn.named_steps['model'].n_neighbors},
])
display(grid_summary)


2026/05/29 14:08:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 14:08:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/29 14:08:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 14:08:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

,model,rmse,mae,r2,best_k
0,Ridge,1.756911,1.359175,0.078942,NaN
1,Lasso,1.757200,1.359605,0.078639,NaN
2,KNN,1.758056,1.338019,0.077741,21.0


In [7]:
from sklearn.model_selection import GridSearchCV

df_class = df.copy()
df_class['high_score'] = (df_class['score'] >= 7).astype(int)
CLASS_TARGET = 'high_score'
X_class = df_class.drop(columns=['score', CLASS_TARGET])
y_class = df_class[CLASS_TARGET]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class, y_class, test_size=0.2, random_state=RANDOM_STATE
)

class_pipe = make_pipeline(
    df_class.drop(columns=['score']),
    CLASS_TARGET,
    get_models('classification')['KNNClassifier']
)

knn_classifier_grid = GridSearchCV(
    class_pipe,
    {
        'model__n_neighbors': list(range(3, 32, 2)),
        'model__weights': ['uniform', 'distance'],
    },
    cv=3,
    scoring='f1',
    n_jobs=1,
)

knn_classifier_grid.fit(Xc_train, yc_train)
print('Best KNN classifier params:', knn_classifier_grid.best_params_)


greedy_classification_rows = []
while mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(nested=True, run_name='05_greedy_all_classification'):
    for name, model in get_models('classification').items():
        if name not in OPTIMIZATION_CLASSIFICATION_MODELS:
            print(f'Skipping {name} in greedy search to keep notebook runtime manageable.')
            continue
        if name in SKIP_OPTIMIZATION:
            print(f'Skipping {name} in greedy search due to computational cost.')
            continue

        tuned_model, search_info = greedy_search(
            make_pipeline(df_class.drop(columns=['score']), CLASS_TARGET, model),
            X_class,
            y_class,
            cv=3,
            scoring='f1',
        )

        preds = tuned_model.predict(Xc_test)
        greedy_classification_rows.append({
            'model': name,
            'search': 'greedy',
            'best_cv_score': search_info['best_score'],
            'best_params': search_info['best_params'],
            **compute_classification_metrics(yc_test, preds),
        })

df_greedy_classification = pd.DataFrame(greedy_classification_rows).sort_values('f1', ascending=False)
display(df_greedy_classification)


optuna_classification_rows = []
while mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(nested=True, run_name='05_optuna_all_classification'):
    for name, model in get_models('classification').items():
        if name not in OPTIMIZATION_CLASSIFICATION_MODELS:
            print(f'Skipping {name} in Optuna to keep notebook runtime manageable.')
            continue
        if name in SKIP_OPTIMIZATION:
            print(f'Skipping {name} in Optuna due to computational cost.')
            continue

        tuned_model, search_info = optuna_study(
            make_pipeline(df_class.drop(columns=['score']), CLASS_TARGET, model),
            X_class,
            y_class,
            n_trials=5,
            cv=3,
            scoring='f1',
        )

        preds = tuned_model.predict(Xc_test)
        optuna_classification_rows.append({
            'model': name,
            'search': 'optuna',
            'best_cv_score': search_info['best_score'],
            'best_params': search_info['best_params'],
            **compute_classification_metrics(yc_test, preds),
        })

df_optuna_classification = pd.DataFrame(optuna_classification_rows).sort_values('f1', ascending=False)
display(df_optuna_classification)

Best KNN classifier params: {'model__n_neighbors': 27, 'model__weights': 'uniform'}
Skipping LogisticRegression in greedy search to keep notebook runtime manageable.


2026/05/29 14:13:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 14:13:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Skipping SVC_linear in greedy search to keep notebook runtime manageable.
Skipping SVC_rbf in greedy search to keep notebook runtime manageable.
Skipping SVC_poly in greedy search to keep notebook runtime manageable.


2026/05/29 14:15:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 14:15:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/29 14:15:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 14:15:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Skipping RandomForestClassifier in greedy search to keep notebook runtime manageable.
Skipping GradientBoostingClassifier in greedy search to keep notebook runtime manageable.
Skipping MLPClassifier_single in greedy search to keep notebook runtime manageable.
Skipping MLPClassifier_multi in greedy search to keep notebook runtime manageable.


...

[I 2026-05-29 14:16:05,339] Trial 0 finished with value: 0.8538767209853416 and parameters: {'var_smoothing': 2.2145280536731464e-09}. Best is trial 0 with value: 0.8538767209853416.
[I 2026-05-29 14:16:05,755] Trial 1 finished with value: 0.8538767209853416 and parameters: {'var_smoothing': 1.2516114645513055e-10}. Best is trial 0 with value: 0.8538767209853416.
[I 2026-05-29 14:16:06,038] Trial 2 finished with value: 0.8538767209853416 and parameters: {'var_smoothing': 5.443174739138011e-09}. Best is trial 0 with value: 0.8538767209853416.
[I 2026-05-29 14:16:06,331] Trial 3 finished with value: 0.8538767209853416 and parameters: {'var_smoothing': 1.027470854565167e-12}. Best is trial 0 with value: 0.8538767209853416.
[I 2026-05-29 14:16:06,596] Trial 4 finished with value: 0.8538767209853416 and parameters: {'var_smoothing': 1.9209143441717461e-07}. Best is trial 0 with value: 0.8538767209853416.
2026/05/29 14:16:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please 

Skipping SVC_linear in Optuna to keep notebook runtime manageable.
Skipping SVC_rbf in Optuna to keep notebook runtime manageable.
Skipping SVC_poly in Optuna to keep notebook runtime manageable.


[I 2026-05-29 14:16:23,093] Trial 0 finished with value: 0.7299178000628729 and parameters: {'n_neighbors': 3, 'weights': 'uniform', 'p': 1}. Best is trial 0 with value: 0.7299178000628729.
[I 2026-05-29 14:16:30,003] Trial 1 finished with value: 0.8205677290971867 and parameters: {'n_neighbors': 27, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 0.8205677290971867.
[I 2026-05-29 14:16:38,059] Trial 2 finished with value: 0.8507841000376065 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 1}. Best is trial 2 with value: 0.8507841000376065.
[I 2026-05-29 14:16:46,054] Trial 3 finished with value: 0.8518272849973961 and parameters: {'n_neighbors': 29, 'weights': 'uniform', 'p': 2}. Best is trial 3 with value: 0.8518272849973961.
[I 2026-05-29 14:16:54,456] Trial 4 finished with value: 0.821283115442256 and parameters: {'n_neighbors': 27, 'weights': 'uniform', 'p': 1}. Best is trial 3 with value: 0.8518272849973961.
2026/05/29 14:16:54 WARNING mlflow.models.mode

Skipping RandomForestClassifier in Optuna to keep notebook runtime manageable.
Skipping GradientBoostingClassifier in Optuna to keep notebook runtime manageable.
Skipping MLPClassifier_single in Optuna to keep notebook runtime manageable.
Skipping MLPClassifier_multi in Optuna to keep notebook runtime manageable.


,model,search,best_cv_score,best_params,accuracy,precision,recall,f1
2,DecisionTreeClassifier,optuna,0.856035,"{'max_depth': 19, 'min_samples_split': 6, 'min...",0.7530,0.750205,0.995374,0.855572
1,KNNClassifier,optuna,0.851827,"{'n_neighbors': 29, 'weights': 'uniform', 'p': 2}",0.7524,0.750154,0.994286,0.855137
0,NaiveBayes,optuna,0.853877,{'var_smoothing': 2.2145280536731464e-09},0.7491,0.750699,0.986122,0.852455


## Greedy and Optuna Optimisation for All Regression Models

This section applies both greedy coordinate-wise search and Optuna to every regression model returned by `get_models('regression')`. Greedy search changes one hyperparameter at a time and keeps the best value before moving to the next parameter. Optuna performs guided sequential search over the same modelling family with continuous or categorical search spaces.

Using both methods for all models makes the comparison broader: simple models such as Linear Regression, Ridge, and Lasso receive small but meaningful tuning spaces, while K-NN, trees, ensembles, SVMs, and neural networks receive capacity and regularisation parameters. The final choice should still consider RMSE, stability, and computational cost, because exhaustive optimisation of every model can be expensive.


In [ ]:
greedy_rows = []
while mlflow.active_run():
    mlflow.end_run()
with mlflow.start_run(nested=True, run_name='05_greedy_all_regression'):
    for name, model in models.items():
        if name not in OPTIMIZATION_REGRESSION_MODELS:
            print(f'Skipping {name} in greedy search to keep notebook runtime manageable.')
            continue
        tuned_model, search_info = greedy_search(
            make_pipeline(df, TARGET, model),
            X_train_df,
            y_train,
            cv=3,
            scoring='neg_mean_squared_error',
        )
        greedy_rows.append({
            'model': name,
            'search': 'greedy',
            'best_cv_score': search_info['best_score'],
            'best_params': search_info['best_params'],
            **evaluate_model(f'{name}_greedy', tuned_model, X_test_df, y_test),
        })

df_greedy = pd.DataFrame(greedy_rows).sort_values('rmse')
display(df_greedy)

optuna_rows = []
while mlflow.active_run():
    mlflow.end_run()
with mlflow.start_run(nested=True, run_name='05_optuna_all_regression'):
    for name, model in models.items():
        if name not in OPTIMIZATION_REGRESSION_MODELS:
            print(f'Skipping {name} in Optuna to keep notebook runtime manageable.')
            continue
        tuned_model, search_info = optuna_study(
            make_pipeline(df, TARGET, model),
            X_train_df,
            y_train,
            n_trials=5,
            cv=3,
            scoring='neg_mean_squared_error',
        )
        optuna_rows.append({
            'model': name,
            'search': 'optuna',
            'best_cv_score': search_info['best_score'],
            'best_params': search_info['best_params'],
            **evaluate_model(f'{name}_optuna', tuned_model, X_test_df, y_test),
        })

df_optuna = pd.DataFrame(optuna_rows).sort_values('rmse')
display(df_optuna)


In [ ]:
# Export dashboard artifacts from Notebook 05
from pathlib import Path

ARTIFACTS_DIR = Path('../artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

if 'df_cv' in globals():
    df_cv.to_csv(ARTIFACTS_DIR / 'cv_results.csv', index=False)
if 'df_greedy' in globals():
    df_greedy.to_csv(ARTIFACTS_DIR / 'greedy_regression_results.csv', index=False)
if 'df_optuna' in globals():
    df_optuna.to_csv(ARTIFACTS_DIR / 'optuna_regression_results.csv', index=False)
if 'df_greedy_classification' in globals():
    df_greedy_classification.to_csv(ARTIFACTS_DIR / 'greedy_classification_results.csv', index=False)
if 'df_optuna_classification' in globals():
    df_optuna_classification.to_csv(ARTIFACTS_DIR / 'optuna_classification_results.csv', index=False)

print('Notebook 05 exported dashboard artifacts:')
print('- cv_results.csv')
print('- greedy/optuna result CSV files when available')
